# Mini-Challenge 1 — Load & Inspect: Saint Mary's patient roster
### Saint Mary's · Thursday morning, 09:45

> *"Here are 200 patients in our care. How many of them carry serious comorbidity load — meaning ≥ 2 chronic conditions?"*

**Time:** 25 min · **Mode:** tandem · **Goal:** first-pass inspection of a real CSV.

---

# Solution — Mini-Challenge 1

## Steps 1-2 — Load and inspect

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Walk up to find the 'data/' folder — works from any notebook location
HERE = Path.cwd()
DATA = next((p / "data" for p in [HERE, *HERE.parents] if (p / "data").exists()), None)
assert DATA is not None, "data/ folder not found"
print("Using DATA =", DATA)

In [ ]:
patients = pd.read_csv(DATA / "patients.csv")

print(patients.shape)
patients.head()

In [ ]:
patients.info()

In [ ]:
patients.describe()

## Step 3 — Count with ≥ 2 chronic conditions

In [ ]:
def count_conditions(s):
    if not isinstance(s, str) or s == "":
        return 0
    return len([c for c in s.split(";") if c])

patients["chronic_count"] = patients["chronic_conditions"].fillna("").apply(count_conditions)

mask = patients["chronic_count"] >= 2
n = mask.sum()
print(f"Patients with >=2 chronic conditions: {n} of {len(patients)} ({n/len(patients):.1%})")

## Fast-Track — Comorbidity rate per insurance

In [ ]:
by_ins = (patients
            .groupby("primary_insurance")
            .agg(mean_chronic=("chronic_count", "mean"),
                 share_high_burden=("chronic_count", lambda s: (s >= 2).mean()),
                 n=("patient_id", "count"))
            .sort_values("share_high_burden", ascending=False))
by_ins